# Build Historical Speed Profile (2021-06-29 → 2026-04-30)

Builds `data/historical_speed_profile_5m/` from `data/vd_station_5m/`.

Uses DuckDB to aggregate per-VD speed percentiles (p10/p50/p90) across all days,
broken down by `day_class` (weekday / weekend / national_holiday) and `peak_class` (peak / nonpeak).

**Per-freeway loop**: the build processes one freeway at a time. The full classify
table for 5 years is ~30 GB; per-freeway it's ~3–4 GB, fitting comfortably in
Colab's 12 GB RAM. Each iteration drops its temp tables before the next CREATE,
and DuckDB's reported memory usage is logged each round so you can verify cleanup.

**Resumable per freeway**: if it crashes on freeway N, completed freeways are
already on Drive. Resume the rest by running the script for each missing freeway
with `--freeway <id>` (no `--force` needed if its directory doesn't exist yet).

> DuckDB spill-to-disk uses `/tmp/duckdb_tmp` (local Colab disk, **not** Drive)
> to avoid latency issues. Typical runtime: **40–90 min** on Colab — slower than
> the old single-pass because each freeway re-scans the parquet glob, but
> memory-safe under the 12 GB limit.

### Run order
1. Cell 1 — Mount Drive + verify
2. Cell 2 — Install uv
3. Cell 3 — Configure
4. Cell 4 — Pre-build check *(optional)*
5. **Cell 5 — Build** *(main, long-running, per-freeway loop)*
6. Cell 6 — Verify output
7. Cell 7 — Post-build Drive sync check

In [ ]:
# ── Cell 1: Mount Drive + verify ──────────────────────────────────────────────
import os, sys, time, json, pathlib, subprocess

from google.colab import drive
print('Mounting Google Drive (force_remount=True)...')
drive.mount('/content/drive', force_remount=True)

DRIVE_ROOT   = pathlib.Path('/content/drive/MyDrive')
PROJECT_ROOT = DRIVE_ROOT / '陽交大高速公路計畫' / 'main'
DATA_ROOT    = PROJECT_ROOT / 'data'
SCRIPT       = PROJECT_ROOT / 'scripts' / 'data' / 'build_historical_profile.py'

# ── 1a. Verify critical paths ──────────────────────────────────────────────────
for p, label in [
    (PROJECT_ROOT,              'project root'),
    (SCRIPT,                    'build script'),
    (DATA_ROOT / 'vd_station_5m', 'station 5m data'),
]:
    if not p.exists():
        raise FileNotFoundError(f'{label} not found at: {p}')
    print(f'  ✓ {label}')

# ── 1b. Sentinel write/read ────────────────────────────────────────────────────
sentinel = PROJECT_ROOT / '.colab_sentinel'
ts = str(time.monotonic())
tag = f'ok:{ts}'
try:
    sentinel.write_text(tag)
    time.sleep(2)
    rb = sentinel.read_text()
    if rb != tag:
        raise AssertionError(f'wrote {tag!r} but read back {rb!r}')
    sentinel.unlink()
    print('✓ Drive write/read sentinel OK')
except Exception as exc:
    try: sentinel.unlink()
    except Exception: pass
    raise RuntimeError(
        f'Drive sentinel check FAILED: {exc}. '
        'Try: Runtime > Factory reset runtime, then re-run this cell.'
    ) from exc

# ── 1c. Show Drive free space ──────────────────────────────────────────────────
stat = os.statvfs('/content/drive')
free_gb  = stat.f_bavail * stat.f_frsize / 1e9
total_gb = stat.f_blocks * stat.f_frsize / 1e9
print(f'  Drive: {free_gb:.1f} GB free / {total_gb:.1f} GB total')
print('Ready.')

In [ ]:
# ── Cell 2: Install uv ────────────────────────────────────────────────────────
result = subprocess.run(['pip', 'install', 'uv', '-q'], capture_output=True, text=True)
if result.returncode != 0:
    print('pip stderr:', result.stderr)
    raise RuntimeError('pip install uv failed')

r = subprocess.run(['uv', '--version'], capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError('uv not available after install — check PATH')
print(r.stdout.strip())

In [ ]:
# ── Cell 3: Configuration ──────────────────────────────────────────────────────
# Adjust dates here to build a subset first.
# MEMORY_LIMIT: Colab free tier has ~12 GB RAM; 8 GB leaves headroom.
# Increase to '10GB' on Colab Pro if you have more RAM.

START_DATE   = '2021-06-29'
END_DATE     = '2026-04-30'
MEMORY_LIMIT = '6GB'
TEMP_DIR     = '/tmp/duckdb_tmp'   # local Colab disk — do NOT set this to a Drive path

def count_parquet(root, pattern):
    p = pathlib.Path(root)
    if not p.exists():
        return 0
    return sum(1 for _ in p.rglob(pattern))

st5_n    = count_parquet(DATA_ROOT / 'vd_station_5m',             'vd_station_*.parquet')
profile_n = count_parquet(DATA_ROOT / 'historical_speed_profile_5m', '*.parquet')

print(f'Date range    : {START_DATE}  →  {END_DATE}')
print(f'Memory limit  : {MEMORY_LIMIT}')
print(f'Temp dir      : {TEMP_DIR}')
print(f'Station 5m    : {st5_n:,} files (input)')
print(f'Profile files : {profile_n:,} existing  →  will be DELETED and rebuilt (--force)')

if st5_n == 0:
    raise RuntimeError('No station_5m files found — ensure build_station_full_range.ipynb has been run first.')

if profile_n > 0:
    print()
    print(f'⚠ WARNING: {profile_n} existing profile file(s) will be overwritten.')
    print('  If you want to abort, stop here and do NOT run Cell 5.')

In [ ]:
# ── Cell 4 (optional): Pre-build check ────────────────────────────────────────
# Shows which freeways exist in station_5m and the expected output layout.
# Skipping this cell does not affect the build.

import duckdb

station_glob = (DATA_ROOT / 'vd_station_5m').as_posix() + '/**/*.parquet'
con_check = duckdb.connect()

print('Freeways in station_5m (sample of first file):')
rows = con_check.sql(
    f"""
    SELECT freeway, COUNT(DISTINCT vdid) AS n_vdid, COUNT(*) AS n_rows
    FROM read_parquet('{station_glob}', hive_partitioning=true)
    WHERE source_date BETWEEN DATE '2024-01-01' AND DATE '2024-01-07'
    GROUP BY freeway
    ORDER BY freeway
    """
).fetchall()

for r in rows:
    print(f'  freeway={r[0]}: {r[1]} VDs, {r[2]:,} rows (sample week 2024-01-01)')

con_check.close()
print('\nPre-flight OK — ready to build.')

In [ ]:
# ── Cell 5: Run build ─────────────────────────────────────────────────────────
# Per-freeway DuckDB loop: discover freeways → for each, classify → aggregate → write.
# --force wipes data/historical_speed_profile_5m/ before rebuilding.
# --temp-dir must be a local path (not Drive) so DuckDB spill-to-disk is fast.
# Each iteration drops temp tables and prints DuckDB memory usage so cleanup is visible.

cmd = [
    'uv', 'run', str(SCRIPT),
    '--start-date',   START_DATE,
    '--end-date',     END_DATE,
    '--data-root',    str(DATA_ROOT),
    '--memory-limit', MEMORY_LIMIT,
    '--temp-dir',     TEMP_DIR,
    '--force',
]
print('Command:', ' '.join(str(c) for c in cmd))
print('=' * 70)
t0 = time.time()

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    cwd=str(PROJECT_ROOT),
    errors='replace',
)

try:
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
except KeyboardInterrupt:
    proc.terminate()
    proc.wait()
    print()
    print('[INTERRUPTED — see resume hint below]')

elapsed = time.time() - t0
print()
print('=' * 70)
print(f'Elapsed: {elapsed / 60:.1f} min  |  Exit code: {proc.returncode}')
if proc.returncode not in (0, None):
    print('WARNING: non-zero exit — check the log above for error details.')
    print('Common fixes:')
    print('  OOM      → lower MEMORY_LIMIT in Cell 3 and re-run')
    print('  No rows  → check START_DATE / END_DATE cover station_5m files')
    print('  Resume   → for each missing freeway dir under historical_speed_profile_5m/,')
    print('             re-run the script with --freeway <id> (no --force needed if dir is absent).')

In [ ]:
# ── Cell 6: Verify output ──────────────────────────────────────────────────────
import random
try:
    import pyarrow.parquet as pq
except ImportError:
    subprocess.run(['pip', 'install', 'pyarrow', '-q'], check=True)
    import pyarrow.parquet as pq

profile_root = DATA_ROOT / 'historical_speed_profile_5m'
freeway_dirs = sorted(profile_root.glob('freeway=*')) if profile_root.exists() else []

print(f'Profile root : {profile_root}')
print(f'Freeway dirs : {len(freeway_dirs)}')

all_files = []
for fd in freeway_dirs:
    files = sorted(fd.glob('*.parquet'))
    total_kb = sum(f.stat().st_size for f in files) / 1024
    print(f'  {fd.name}: {len(files)} file(s), {total_kb/1024:.1f} MB')
    all_files.extend(files)

# ── Spot-read 5 random files ───────────────────────────────────────────────────
problems = []
if all_files:
    print()
    print('Spot-check (up to 5 random profile files):')
    sample = random.sample(all_files, min(5, len(all_files)))
    for f in sample:
        try:
            sz_kb = f.stat().st_size / 1024
            meta  = pq.ParquetFile(str(f)).metadata
            nrows = meta.num_rows
            cols  = meta.schema.names
            ok    = sz_kb > 1 and nrows > 0 and 'p50_speed_x10' in cols
        except Exception as exc:
            sz_kb, nrows, ok = 0, f'READ ERROR: {exc}', False
        icon = '✓' if ok else '✗'
        print(f'  {icon} {f.parent.name}/{f.name}: {sz_kb:.0f} KB, {nrows:,} rows')
        if not ok:
            problems.append(str(f))
    if problems:
        print(f'  ⚠ {len(problems)} file(s) have issues')
    else:
        print('  ✓ All spot-checked files look valid')

# ── Build report ──────────────────────────────────────────────────────────────
report_path = DATA_ROOT / 'metadata' / 'historical_profile_report.json'
if report_path.exists():
    rpt = json.loads(report_path.read_text())
    print()
    print('Build report:')
    print(f'  Range         : {rpt.get("range", {})}')
    print(f'  Profile rows  : {rpt.get("rows_total", 0):,}')
    print(f'  Input rows    : {rpt.get("input_rows_used", 0):,}')
    dc = rpt.get('day_class_days', {})
    print(f'  Day classes   : weekday={dc.get("weekday",0)}, weekend={dc.get("weekend",0)}, national_holiday={dc.get("national_holiday",0)}')
    print(f'  Classify sec  : {rpt.get("classify_sec", "?")}')
    print(f'  Aggregate sec : {rpt.get("aggregate_sec", "?")}')
    print(f'  Write sec     : {rpt.get("write_sec", "?")}')
else:
    print('(historical_profile_report.json not found yet)')

In [ ]:
# ── Cell 7: Post-build Drive sync check ───────────────────────────────────────
# Confirms Drive is still flushing writes after the build.
# If this fails, wait 2–3 minutes and re-run this cell before closing the session.

sentinel2 = PROJECT_ROOT / '.colab_profile_done'
ts2 = str(time.monotonic())
tag2 = f'done:{ts2}:{len(all_files)}x_profile'

try:
    sentinel2.write_text(tag2)
    time.sleep(3)
    rb2 = sentinel2.read_text()
    if rb2 != tag2:
        raise AssertionError(f'wrote {tag2!r}, read back {rb2!r}')
    sentinel2.unlink()
    print('✓ Post-build Drive write/read verified')
    print(f'✓ {len(all_files):,} profile parquet file(s) on Drive')
    print()
    print('All done!')
except Exception as exc:
    try: sentinel2.unlink()
    except Exception: pass
    print(f'✗ Post-build Drive check FAILED: {exc}')
    print('  Wait 2–3 min (Drive may still be flushing), then re-run this cell.')
    print('  If it keeps failing, check Drive storage quota and connectivity.')